# Retainly Research Validation: Baseline Models vs Multi-Agent Attrition Intelligence

This notebook evaluates whether Retainly's multi-agent workflow improves attrition prediction and decision-support quality over standard single-model ML baselines.


## Research Objective
To evaluate whether Retainly's multi-agent workflow improves attrition prediction and HR usability compared with normal single-pipeline ML baselines.


## Why This Comparison Matters
A normal ML model outputs predictions. Retainly performs dataset validation, model selection, threshold tuning, explainability, fairness review, and HR action planning.


## Datasets Setup
The notebook expects labeled validation datasets placed locally in `research_datasets/`. If files are missing, the comparison script will clearly report which ones need to be added.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scripts.run_dataset_comparison import load_available_datasets, detect_target_column, normalize_target, prepare_features, run_baseline_logistic_regression, run_baseline_random_forest, run_baseline_gradient_boosting, run_retainly_multi_agent_pipeline, compute_metrics, compute_topk_metrics, audit_fairness, extract_top_drivers, generate_hr_actions, run_comparison, save_outputs
root = Path('..').resolve()
out = root / 'research_outputs'
datasets = load_available_datasets(root / 'research_datasets')
datasets


## Methods Compared
The comparison includes three baseline models and the Retainly multi-agent pipeline.


In [ ]:
method_table = pd.DataFrame([
    {'Approach': 'Baseline Logistic Regression', 'Models used': 'LogisticRegression', 'Preprocessing': 'Basic numeric/categorical preprocessing', 'Threshold strategy': 'Fixed 0.5', 'Explainability': 'No', 'Fairness audit': 'No', 'HR action output': 'No'},
    {'Approach': 'Baseline Random Forest', 'Models used': 'RandomForestClassifier', 'Preprocessing': 'Basic numeric/categorical preprocessing', 'Threshold strategy': 'Fixed 0.5', 'Explainability': 'No', 'Fairness audit': 'No', 'HR action output': 'No'},
    {'Approach': 'Baseline Gradient Boosting', 'Models used': 'GradientBoostingClassifier', 'Preprocessing': 'Basic numeric/categorical preprocessing', 'Threshold strategy': 'Fixed 0.5', 'Explainability': 'No', 'Fairness audit': 'No', 'HR action output': 'No'},
    {'Approach': 'Retainly Multi-Agent Pipeline', 'Models used': 'Multiple models + leaderboard', 'Preprocessing': 'Governed preprocessing with leakage/ID/sensitive-field removal', 'Threshold strategy': 'Tuned 0.20-0.80', 'Explainability': 'Yes', 'Fairness audit': 'Yes', 'HR action output': 'Yes'},
] )
method_table


## Retainly Multi-Agent Workflow: What Each Agent Does

**Project Manager Agent**: coordinates the workflow and ensures each step runs in sequence.
- Purpose: Coordinates the workflow and ensures each step runs in sequence.
- Input: Raw dataset metadata.
- Processing: Creates and tracks the execution plan.
- Output: Execution plan and pipeline trace.
- Why it improves the pipeline: Prevents steps from being skipped and makes the run auditable.

**Smart Column Mapper Agent**: detects target, ID columns, sensitive fields, business fields, numeric/categorical features.
- Purpose: Detects target, ID columns, sensitive fields, business fields, numeric/categorical features.
- Input: Raw column names and dataset profile.
- Processing: Infers target, identity, sensitive, and business columns.
- Output: Clean column map.
- Why it improves the pipeline: Makes the pipeline work across different HR datasets.

**Data Analyst Agent**: data validation, missing values, class balance, leakage checks, basic EDA.
- Purpose: Data validation, missing values, class balance, leakage checks, basic EDA.
- Input: Cleaned dataset and detected mapping.
- Processing: Summarizes missingness, balance, and leakage risk.
- Output: EDA summary and data quality note.
- Why it improves the pipeline: Prevents poor-quality data and leakage from corrupting model results.

**ML Engineer Agent**: trains and compares multiple models, handles imbalance, tunes thresholds, selects best model.
- Purpose: Trains and compares multiple models, handles imbalance, tunes thresholds, selects best model.
- Input: Prepared features and target.
- Processing: Builds a leaderboard and tunes thresholds by practical score.
- Output: Selected model, threshold, and ranked evaluation.
- Why it improves the pipeline: Better than using a single default model.

**Explainability Agent**: extracts top predictive drivers using feature importance/permutation importance.
- Purpose: Extracts top predictive drivers using feature importance/permutation importance.
- Input: Trained model and evaluation data.
- Processing: Computes top drivers for interpretation.
- Output: Top predictive drivers.
- Why it improves the pipeline: Converts predictions into understandable reasons.

**Fairness Auditor Agent**: audits sensitive-group differences.
- Purpose: Audits sensitive-group differences.
- Input: Predictions and sensitive attributes.
- Processing: Summarizes disparities across sensitive groups.
- Output: Fairness status and disparity summary.
- Why it improves the pipeline: Adds responsible AI review that baseline ML lacks.

**Insights Agent**: turns model outputs into HR action recommendations.
- Purpose: Turns model outputs into HR action recommendations.
- Input: Top drivers, risk ranking, and fairness notes.
- Processing: Converts results into HR actions.
- Output: HR recommendations and actions.
- Why it improves the pipeline: Makes the system useful to HR, not just data scientists.


In [ ]:
if not datasets:
    print('No benchmark datasets found. Place at least one CSV in research_datasets/.')
else:
    comp = run_comparison(datasets=datasets, output_dir=out, seed=42, save=True)
    save_outputs(comp, out)
    print('Saved outputs to', out)


## Dataset Setup
The table below shows the rows, columns, target column, attrition rate, fairness fields detected, and business fields detected.


In [ ]:
if (out / 'dataset_comparison_results.csv').exists():
    results = pd.read_csv(out / 'dataset_comparison_results.csv')
    dataset_overview = results[['dataset','approach','rows','columns','target_column','attrition_rate']].drop_duplicates(subset=['dataset']).sort_values('dataset')
    dataset_overview
else:
    print('Run the comparison cell first.')


## Experimental Fairness
The comparison keeps the same split, same target, same random seed, and same test set. Retainly removes sensitive/leakage fields from training, while baselines use the same cleaned target and same split.


## Model Training and Evaluation
The cells below show the outputs of the benchmark run.


In [ ]:
if (out / 'dataset_comparison_results.csv').exists():
    results = pd.read_csv(out / 'dataset_comparison_results.csv')
    display_cols = ['dataset','approach','accuracy','precision','recall','f1','roc_auc','pr_auc','recall_at_top_10_percent','recall_at_top_20_percent','lift_at_top_10_percent','lift_at_top_20_percent','selected_model','selected_threshold','fairness_status','max_fairness_disparity','predictive_score','usability_score','final_project_score']
    display_cols = [c for c in display_cols if c in results.columns]
    results[display_cols].sort_values(['dataset','approach'])
else:
    print('Run the comparison cell first.')


In [ ]:
if (out / 'dataset_comparison_results.csv').exists():
    results = pd.read_csv(out / 'dataset_comparison_results.csv')
    avg = results.groupby('approach')[['accuracy','precision','recall','f1','roc_auc','pr_auc','recall_at_top_10_percent','recall_at_top_20_percent','lift_at_top_10_percent','lift_at_top_20_percent','final_project_score']].mean(numeric_only=True).round(3)
    avg
else:
    print('Run the comparison cell first.')


## Graphs


In [ ]:
chart = out / 'dataset_comparison_core_metrics.png'
if chart.exists():
    img = plt.imread(chart)
    plt.figure(figsize=(12,6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Core metrics chart not found yet.')


In [ ]:
chart = out / 'dataset_comparison_topk_metrics.png'
if chart.exists():
    img = plt.imread(chart)
    plt.figure(figsize=(12,6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Top-k chart not found yet.')


In [ ]:
chart = out / 'dataset_comparison_final_score.png'
if chart.exists():
    img = plt.imread(chart)
    plt.figure(figsize=(12,6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Final score chart not found yet.')


## Explainability Output
Top drivers for Retainly are summarized in the output tables.


In [ ]:
if (out / 'top_drivers_summary.csv').exists():
    pd.read_csv(out / 'top_drivers_summary.csv').head(20)
else:
    print('Top driver summary not found yet.')


## Fairness / Responsible AI Review
Department and JobRole are treated as business segments, not fairness fields. The fairness review focuses on sensitive attributes only.


In [ ]:
if (out / 'fairness_summary.csv').exists():
    pd.read_csv(out / 'fairness_summary.csv')
else:
    print('Fairness summary not found yet.')


## HR Action Generation
The Retainly pipeline generates HR-oriented actions from the top drivers and risk ranking. Baseline models do not generate these outputs.


In [ ]:
if (out / 'hr_actions_summary.csv').exists():
    pd.read_csv(out / 'hr_actions_summary.csv').head(20)
else:
    print('HR actions summary not found yet.')


## Discussion
Retainly may not win every individual metric, but it should perform better on attrition-focused and ranked-list metrics. It also adds explainability, fairness review, and action planning.


## Final Conclusion
The multi-agent Retainly workflow provides stronger predictive performance and greater decision-support value than a normal baseline ML workflow, while also adding explainability, fairness review, and HR action planning.


## Connection to Website
The notebook validates the approach on labeled datasets. The website applies the validated/pretrained approach to current employee data that may not have Attrition.
